In [ ]:
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# ==============================
# 1. 상태 정의
# ==============================
class GraphState(TypedDict):
    messages: Annotated[List, add_messages]

# ==============================
# 2. 툴 정의
# ==============================
@tool
def add(a: int, b: int) -> int:
    """두 수를 더합니다"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 수를 곱합니다"""
    return a * b

# ==============================
# 3. LLM + Agent + ToolNode
# ==============================
llm = ChatOpenAI(model="gpt-4o-mini")
tools = [add, multiply]

# Agent 생성
prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful AI that can call tools."),
     ("user", "{input}"),
     ("placeholder", "{agent_scratchpad}")]
)
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

def tool_node(state: GraphState):
    # Agent 실행 → 필요시 tool 호출
    response = agent_executor.invoke({"input": state["messages"][-1].content})
    return {"messages": [AIMessage(content=str(response))]}

# ==============================
# 4. 사람에게 묻는 노드
# ==============================
def ask_human(state: GraphState):
    return {"messages": [("system", "❓ 사람이 직접 결정할 차례입니다!")]}
    # 실제 프로덕션에서는 UI에서 입력 기다림

# ==============================
# 5. 대화 기록 관리 (요약)
# ==============================
def summarize_if_long(state: GraphState):
    msgs = state["messages"]
    if len(msgs) > 6:
        # 요약 생성
        summary = f"요약: 이전 {len(msgs)}개의 대화가 있었습니다."
        # 최근 2개 대화만 유지
        new_msgs = msgs[-2:]
        return {"messages": [("system", summary)] + new_msgs}
    return state

# ==============================
# 6. 노드 정의
# ==============================
def naive_rag(state: GraphState):
    # 간단히 RAG 흉내 → 실제론 VectorStore 호출
    query = state["messages"][-1].content
    return {"messages": [("system", f"[RAG] '{query}' 관련 문서 검색 결과")]}

def fan_out(state: GraphState):
    return {"messages": [("system", "Fan-out 노드 → 여러 경로로 분기")]}

def fan_in(state: GraphState):
    return {"messages": [("system", "Fan-in 노드 → 여러 결과를 합침")]}

def final_node(state: GraphState):
    return {"messages": [("system", "최종 END 도착")]}

# ==============================
# 7. 그래프 구성
# ==============================
graph = StateGraph(GraphState)

graph.add_node("tool", tool_node)
graph.add_node("ask_human", ask_human)
graph.add_node("summarize", summarize_if_long)
graph.add_node("rag", naive_rag)
graph.add_node("fan_out", fan_out)
graph.add_node("fan_in", fan_in)
graph.add_node("final", final_node)

graph.add_edge(START, "summarize")

# 조건 분기 (conditional branching)
graph.add_conditional_edges(
    "summarize",
    lambda s: "rag" if "검색" in s["messages"][-1].content else "tool",
    {"rag": "rag", "tool": "tool"}
)

# Fan-out → 병렬 분기
graph.add_edge("rag", "fan_out")
graph.add_edge("tool", "fan_out")

# Fan-in → 결과 합침
graph.add_edge("fan_out", "fan_in")

# 사람에게 묻기 → 의견 필요시
graph.add_edge("fan_in", "ask_human")

# 최종 종료
graph.add_edge("ask_human", "final")
graph.add_edge("final", END)

# ==============================
# 8. 실행기 빌드
# ==============================
app = graph.compile()

# ==============================
# 9. 테스트 실행
# ==============================
print("=== 첫 실행 ===")
state = app.invoke({"messages": [HumanMessage(content="3과 5를 더해줘")]})
print(state)

print("\n=== 검색 요청 실행 ===")
state = app.invoke({"messages": [HumanMessage(content="AI 역사 검색")]})
print(state)

print("\n=== 대화 길이 초과 요약 테스트 ===")
long_dialog = {"messages": [HumanMessage(content=f"msg {i}") for i in range(8)]}
state = app.invoke(long_dialog)
print(state)


In [ ]:
import gradio as gr
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# ==============================
# 1. 상태 정의
# ==============================
class GraphState(TypedDict):
    messages: Annotated[List, add_messages]

# ==============================
# 2. 툴 정의
# ==============================
@tool
def add(a: int, b: int) -> int:
    """두 수를 더합니다"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 수를 곱합니다"""
    return a * b

# ==============================
# 3. LLM + Agent + ToolNode
# ==============================
llm = ChatOpenAI(model="gpt-4o-mini")
tools = [add, multiply]

prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful AI that can call tools."),
     ("user", "{input}"),
     ("placeholder", "{agent_scratchpad}")]
)
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False)

def tool_node(state: GraphState):
    response = agent_executor.invoke({"input": state["messages"][-1].content})
    return {"messages": [AIMessage(content=str(response))]}

# ==============================
# 4. 사람에게 묻는 노드
# ==============================
def ask_human(state: GraphState):
    return {"messages": [("system", "❓ 사람이 직접 결정할 차례입니다!")]}  

# ==============================
# 5. 대화 기록 관리 (요약)
# ==============================
def summarize_if_long(state: GraphState):
    msgs = state["messages"]
    if len(msgs) > 6:
        summary = f"요약: 이전 {len(msgs)}개의 대화가 있었습니다."
        new_msgs = msgs[-2:]  # 최근 2개만 유지
        return {"messages": [("system", summary)] + new_msgs}
    return state

# ==============================
# 6. 노드 정의
# ==============================
def naive_rag(state: GraphState):
    query = state["messages"][-1].content
    return {"messages": [("system", f"[RAG] '{query}' 관련 문서 검색 결과")]}

def fan_out(state: GraphState):
    return {"messages": [("system", "Fan-out 노드 → 여러 경로로 분기")]}

def fan_in(state: GraphState):
    return {"messages": [("system", "Fan-in 노드 → 여러 결과를 합침")]}

def final_node(state: GraphState):
    return {"messages": [("system", "최종 END 도착")]}

# ==============================
# 7. 그래프 구성
# ==============================
graph = StateGraph(GraphState)

graph.add_node("tool", tool_node)
graph.add_node("ask_human", ask_human)
graph.add_node("summarize", summarize_if_long)
graph.add_node("rag", naive_rag)
graph.add_node("fan_out", fan_out)
graph.add_node("fan_in", fan_in)
graph.add_node("final", final_node)

graph.add_edge(START, "summarize")
graph.add_conditional_edges(
    "summarize",
    lambda s: "rag" if "검색" in s["messages"][-1].content else "tool",
    {"rag": "rag", "tool": "tool"}
)
graph.add_edge("rag", "fan_out")
graph.add_edge("tool", "fan_out")
graph.add_edge("fan_out", "fan_in")
graph.add_edge("fan_in", "ask_human")
graph.add_edge("ask_human", "final")
graph.add_edge("final", END)

app = graph.compile()

# ==============================
# 8. Gradio 인터페이스
# ==============================
def chat_fn(message, history):
    # 메시지 기록을 GraphState 형식으로 변환
    state = {"messages": [HumanMessage(content=m[0]) if i % 2 == 0 else AIMessage(content=m[1]) 
                          for i, m in enumerate(history)] + [HumanMessage(content=message)]}
    result = app.invoke(state)

    # 마지막 메시지 추출
    last_msg = result["messages"][-1]
    return str(last_msg.content)

examples = [
    ["3과 5를 더해줘"],
    ["10과 20을 곱해줘"],
    ["AI 역사 검색"],
    ["대화가 길어지면 요약해줘"]
]

with gr.Blocks() as demo:
    gr.Markdown("## 🧩 LangGraph 종합 테스트")
    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="질문 입력")
    btn = gr.Button("보내기")
    ex = gr.Examples(examples, inputs=msg)

    def respond(user_message, chat_history):
        response = chat_fn(user_message, chat_history)
        chat_history.append((user_message, response))
        return "", chat_history

    btn.click(respond, [msg, chatbot], [msg, chatbot])
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch()
